### Import Libraries and Set Config

In [ ]:
from ultralytics import YOLO
import ultralytics
from pathlib import Path
from IPython.display import Image, display
from pathlib import Path
import os
import pandas as pd
import numpy as np
import torch

# Disable MLflow callback to prevent tracking errors
ultralytics.settings.update({'mlflow': False})

# Path to the dataset YAML file from Notebook 02
DATA_CONFIG = Path("Weed-crop RGB dataset/Corn_augmented/corn_augmented.yaml")
MODEL_NAME = "yolo11n.pt"
OUTPUT_DIR = Path("runs/corn_baseline_yolov11n")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check available device
if torch.cuda.is_available():
    DEVICE = 0
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = 'cpu'
    print("⚠ No GPU detected. Training will use CPU (slower).")
    print("   Consider installing CUDA-enabled PyTorch for faster training.")

print(f"Device for training: {DEVICE}")

### Load Model and Train

In [ ]:
# Load model and start training
model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(DATA_CONFIG),
    epochs=100,
    imgsz=640,
    batch=4,
    # workers=0,
    name="training_results",
    project=OUTPUT_DIR,
    device=DEVICE,   # automatically uses GPU if available, otherwise CPU
    patience=20 # early stopping
)


# # Load model and start training
# model = YOLO(MODEL_NAME)
# train_results = model.train(
#     data=str(DATA_CONFIG),
#     epochs=100,
#     imgsz=640,
#     batch=4,
#     # workers=0,
#     name="training_results",
#     project=OUTPUT_DIR,
#     device=0,   # use GPU 0 if available
#     patience=20 # early stopping
# )

### Evaluate Best Model (defined on val dataset) on Test Dataset

In [ ]:
BEST_MODEL_PATH = Path(OUTPUT_DIR) / 'training_results' / 'weights' / 'best.pt'

print(f"Loading best model from: {BEST_MODEL_PATH}")

#eval model that reached best results on val dataste
final_model = YOLO(BEST_MODEL_PATH)

print("\n--- Final Evaluation on TEST Dataset ---")

metrics = final_model.val(
    data=DATA_CONFIG, 
    split='test',      
    imgsz=640
)

print(metrics)
print("\n--- Results on Test Set ---")
print(f"Mean Average Precision (mAP50-95): {metrics.box.map:.4f}")
print(f"mAP50 (Test): {metrics.box.map50:.4f}")
print(f"Recall (Test): {metrics.box.mr.mean():.4f}")

### Visualize Training and Validation Curves

In [ ]:
RUNS_BASE_DIR = Path(OUTPUT_DIR) / 'training_results'

results_plot = RUNS_BASE_DIR/'results.png'

if results_plot.exists():
    print("Found training curve plot:")
    display(Image(filename=str(results_plot))) 
else:
    print("'results.png' not found. Check if training ran successfully.")

### Save Model Export

In [ ]:
# Export trained model weights for future notebooks
export_path = OUTPUT_DIR / "corn_yolo11n_baseline.pt"
model.save(export_path)
print(f"Model saved to: {export_path}")

### Report

In [ ]:
map50_95 = metrics.box.map  
map50 = metrics.box.map50  
precision = metrics.box.mp.mean() if metrics.box.mp.size > 0 else 0  # Mean Precision
recall = metrics.box.mr.mean() if metrics.box.mr.size > 0 else 0    # Mean Recall

df_general = pd.DataFrame({
    'Metric': ['mAP@0.5', 'mAP@0.5:0.95', 'Precision (P)', 'Recall (R)'],
    'Value': [map50, map50_95, precision, recall]
})

df_general.set_index('Metric', inplace=True)

print("--- Overall Metrics ---")
display(df_general)

In [ ]:

ap50_arr = metrics.box.ap50.flatten()
ap_arr = metrics.box.ap.flatten()
class_names_list = metrics.names
nc = len(class_names_list)
nc_evaluated = len(ap50_arr) 

data_list = []
for i in range(nc_evaluated): 
    data_list.append({
        'Class': class_names_list[i], 
        'mAP@0.5': ap50_arr[i],
        'mAP@0.5:0.95': ap_arr[i]
    })

df_class_wise = pd.DataFrame(data_list)

df_class_wise = df_class_wise[
    (df_class_wise['mAP@0.5'] > 0) | (df_class_wise['mAP@0.5:0.95'] > 0)
]
df_class_wise.set_index('Class', inplace=True)

print("\n--- Class-Wise Details ---")
display(df_class_wise)